# 00 · Explore profiles
Well-level Cell Painting profiles (Zenodo 10.5281/zenodo.17067683). Exploration only — all production logic lives in `src/cpsa/`.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, numpy as np
pd.set_option('display.width', 200)
from cpsa import RESULTS, PROCESSED
from cpsa.data.load_profiles import load_raw, feature_columns
raw = load_raw('cpcnn')
print(raw.shape, len(feature_columns(raw)))
raw[['Metadata_Plate','Metadata_Compound','Metadata_Concentration','Metadata_Count_Cells']].describe(include='all')

Wells per compound, controls, and the plate / batch layout (each compound spans 13–16 plates from two production batches).

In [ ]:
t = raw[(raw.Metadata_Compound!='DMSO') & raw.Metadata_OASIS_ID.notna()]
print('DMSO wells:', (raw.Metadata_Compound=='DMSO').sum(), '| compounds with OASIS id:', t.Metadata_OASIS_ID.nunique())
print(t.groupby('Metadata_OASIS_ID').Metadata_Plate.nunique().describe())
raw.Metadata_source.value_counts()

Cell count vs concentration across all compounds (dose-dependent cytotoxicity is what the baseline can exploit).

In [ ]:
import matplotlib.pyplot as plt
dmso = raw[raw.Metadata_Compound=='DMSO'].groupby('Metadata_Plate').Metadata_Count_Cells.median()
t = t.assign(ratio=t.Metadata_Count_Cells / t.Metadata_Plate.map(dmso))
t.groupby(pd.qcut(t.Metadata_Concentration, 8, duplicates='drop'), observed=True).ratio.median().plot(marker='o'); plt.ylabel('cell count / plate DMSO median'); plt.xticks(rotation=45)